# 🧠 LeetCode 22: Generate Parentheses

**Pattern:** Backtracking (Recursive Stacks)

- **Created:** 2026-02-28
- **Focus:** Validating state conditionally as we trace recursive branching paths
- **Tags:** `string` | `dynamic-programming` | `backtracking` | `medium` | `citi-prep`

## 📖 Problem Statement

Given `n` pairs of parentheses, write a function to *generate all combinations of well-formed parentheses*.

### Example 1:
- **Input:** `n = 3`
- **Output:** `["((()))","(()())","(())()","()(())","()()()"]`

### Example 2:
- **Input:** `n = 1`
- **Output:** `["()"]`

## 🧠 Mental Model & Intuition

This perfectly introduces **Backtracking**. 

Imagine you are walking through a maze. At every step, you can choose to go Left (Add an open parenthesis `(`) or go Right (Add a closed parenthesis `)`). 

You logically trace every single possible path. However, there are **Rules (Constraints)**:
1. You only have `n` left turns (Open brackets) available in your pocket.
2. You can never take a right turn (Close bracket) unless you have *already* taken a left turn. Closing without opening first makes the string malformed.

When you hit a dead end, or run out of brackets, you step *backwards* (Backtrack) and try the other route.

## 🐢 Brute Force Approach

Generate literally *every* possible combination of $2n$ characters of `(` and `)`. Then, run the $O(N)$ string validator on every single one of them and throw away the junk ones.

```python
def generateParenthesisBrute(n):
    def is_valid(p_string):
        balance = 0
        for char in p_string:
            if char == '(': balance += 1
            else: balance -= 1
            if balance < 0: return False
        return balance == 0
        
    def generate(p_string):
        if len(p_string) == 2 * n:
            if is_valid(p_string): res.append(p_string)
            return
        generate(p_string + '(')
        generate(p_string + ')')
        
    res = []
    generate("")
    return res
```
**Time:** $O(2^{2n} \times n)$ (Generates absolutely horrific unviable paths) | **Space:** $O(2^{2n})$

In [ ]:
# Optimal Backtracking Approach
def generateParenthesis(n: int) -> list[str]:
    # We use a Stack to track our current "Active Path"
    stack = []
    res = []
    
    def backtrack(openN: int, closedN: int):
        # The Base Case: If we used all components, our path is a success.
        if openN == n and closedN == n:
            res.append("".join(stack))
            return
            
        # Choice 1: Can we add an Open Parenthesis?
        if openN < n:
            stack.append("(")          # 1. Take the step
            backtrack(openN + 1, closedN) # 2. Explore that path completely
            stack.pop()                # 3. BACKTRACK (step backwards from that path)
            
        # Choice 2: Can we add a Closed Parenthesis?
        # We can ONLY close if we have more open than closed currently.
        if closedN < openN:
            stack.append(")")          # 1. Take the step
            backtrack(openN, closedN + 1) # 2. Explore
            stack.pop()                # 3. BACKTRACK

    backtrack(0, 0)
    return res

print("Pairs for n=2: ", generateParenthesis(2))
print("Pairs for n=3: ", generateParenthesis(3))

## ⏱️ Complexity Analysis

- **Time Complexity:** $O(\frac{4^n}{\sqrt{n}})$. This evaluates to the $n^{th}$ Catalan number. By strictly enforcing constraints *before* doing the recursion, we instantly kill all dead-ends. We only generate mathematical exact valid sequences.
- **Space Complexity:** $O(N)$ for the recursion depth / stack memory. It will never nest deeper than $2n$ recursive calls.

## 🎤 Interview Q&A

### Q1: Why do we `stack.pop()` immediately after the `backtrack()` call?
**Answer:** Because we share a single `stack` list in memory. When `backtrack(openN + 1 ...)` finishes exploring all millions of possibilities down the "Left" branch, control is returned to the current function scope. We must remove the `(` we added so we can now try adding a `)` to explore the "Right" branch from a clean slate. This explicit state cleanup is the definition of *Backtracking*.

---

### Q2: Could we do this by passing strings instead of using a list stack?
**Answer:** Yes! `backtrack(openN+1, closedN, current_str + "(")`. Since strings in Python are immutable, adding a string creates a brand new string assigned safely to that deeper recursive scope level. You wouldn't even need to `.pop()` or cleanup because the parent scope's string never changed. **However**, creating thousands of new string objects in memory is fundamentally slower and less memory-efficient than repeatedly appending/popping a single Array array in place.

## 📚 Key Terminology

| Term | Definition | Example |
|---|---|---|
| **Backtracking** | Exploring all potential solutions path-by-path, returning back a level when a path fails or concludes. | `DFS + State Cleanup` |
| **State Cleanup** | The required `.pop()` step ensuring shared structures are restored. | `stack.pop()` |
| **Constraint Enforcement** | Only executing recursion when conditions permit it, rather than recursing blindly. | `if closedN < openN:` |

## 💼 The Citi Narrative

**Context:** Generating valid synthetic XML configurations for Chaos Monkey testing.

**Scenario:** Our Chaos Monkey test suite needed to purposefully barrage the trade-matching engine with hundreds of thousands of varied, but *structurally valid*, heavily nested XML schema bodies to stress-test the parsing threshold of the heap space.

**Impact:** Randomly concatenating `<a>` and `</a>` tags resulted in immediate failures at the gateway firewall because the XML was outright malformed. Implementing a Backtracking algorithm generating the N-th Catalan sequences allowed us to dynamically spool millions of uniquely-shaped, perfectly valid arbitrary XML node hierarchies directly into the target microservice, discovering a critical memory leak.

In [ ]:
# EXERCISE: See the string-passing variant (No .pop required)
def generateStringVariant(n):
    res = []
    def backtrack(op, cl, path):
        if op == n and cl == n:
            res.append(path)
            return
        if op < n:
            backtrack(op + 1, cl, path + '(') # Create new string magically
        if cl < op:
            backtrack(op, cl + 1, path + ')')
            
    backtrack(0, 0, "")
    return res
    
print("Much cleaner, but less 'Enterprise' memory-efficient than Array manipulation.")

## 🎯 Summary: Key Takeaways

### The Pattern
**Backtracking** — Decision-tree branching that instantly aborts paths breaking condition logic.

### When to Use It
- ✅ Generating complex, strictly well-formed permutations for testing.
- ✅ Exploring localized state spaces with rigid correctness constraints.
- ❌ **Don't use when:** Simple combinatorial generation where all permutations are equally valid.

### Complexity
| Approach | Time | Space |
|----------|------|-------|
| Brute Force | $O(2^{2n} \times n)$ | $O(2^{2n})$ |
| Optimal | $O(4^n / \sqrt{n})$ | $O(N)$ |

### Interview Confidence Checklist
- [ ] Can explain the brute force and why it fails
- [ ] Can state the pattern name and core insight in one sentence
- [ ] Can write the optimal solution from memory
- [ ] Can state time and space complexity with justification
- [ ] Can name at least one real-world / Citi application

---

**"Simplicity and clarity is Gold."** — Sean's Study Mantra

Master **Backtracking** and you've mastered one of the most common patterns in data engineering interviews. 🚀